In [ ]:
# git clone https://github.com/josephreplogle/guide_calling

In [ ]:
import os
import sys
from multiprocessing import Pool, cpu_count

sys.path.append('/home/users/ppdu/software/guide_calling')
import guide_calling

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances

import scanpy as sc

import seaborn as sns
import matplotlib.pyplot as plt
import colorcet as cc

import warnings
from glob import glob
from natsort import natsorted
from tqdm import tqdm

In [ ]:
sys.path.append('../auxiliary_scripts')
from graphics import dotplot
from io import read_gmt, write_gmt

In [ ]:
adata = sc.read_h5ad('../results/adata_proc_singlets.h5ad')
cancer = adata[adata.obs['major_celltype'] == 'cancer'].copy()
cancer

In [ ]:
sc.pp.highly_variable_genes(cancer)
cancer.var['highly_variable'] = cancer.var['highly_variable'] & ~ cancer.var['syn']
cancer.var['highly_variable'].sum()

In [ ]:
sc.tl.pca(cancer,use_highly_variable=True)
sc.pp.neighbors(cancer)
sc.tl.umap(cancer)

# sgRNA calling

In [ ]:
read_table = pd.read_table('../data/MATCH_barcode_table.txt.gz')
read_table

In [ ]:
cell_barcodes = pd.DataFrame({'cell_barcode': read_table['cell'].unique()})
cell_barcodes

In [ ]:
samples = [
    'GEM-1', 
    'GEM-2', 
    'GEM-3', 
    'GEM-4', 
]

In [ ]:
res = {}

outdir = '../results/guide_calls_sgRNA'
os.makedirs(outdir, exist_ok=True)

gbc_reads = read_table[['sgRNA', 'cell', '10X_UMI']].copy() # for sgRNA only
gbc_reads.columns = ['guide_identity', 'cell_barcode', 'UMI']

guide_counts = gbc_reads['guide_identity'].value_counts()
guides = guide_counts.index # use all guides

for sample in samples:
    print(sample)
    
    local_reads = gbc_reads.loc[gbc_reads['cell_barcode'].str.endswith(sample)]
    local_cells = cell_barcodes.loc[cell_barcodes['cell_barcode'].str.endswith(sample)]
    gbc_table = guide_calling.capture_reads(local_reads, local_cells)
    
    pop = pd.DataFrame()
    for guide in tqdm(guides):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                pop = pop.append(guide_calling.MixedModelCall(guide, gbc_table, sample, directory=None, max_iter=100))
        except:
            print('could not call ' + guide)
            pass
    res[sample] = pop.copy()
    pop.to_csv(os.path.join(outdir, sample+'_cell_identities.txt'), sep='\t', index=False)

In [ ]:
guide_calls = pd.concat(res)
guide_calls = guide_calls.reset_index().drop(['level_0', 'level_1', 'read_count'], axis=1)
guide_calls

In [ ]:
exemplars = []

for key, value in tqdm(valid.groupby('cell_barcode')):

    value = value.sort_values('UMI_count', ascending=False)
    diff = np.diff(value['UMI_count'])
    if len(diff) == 0:
        diff = [0]
    # most abundant sgRNA has more than X UMIs and difference of Y from next most abundant
    if (value.iloc[0]['UMI_count'] >= 5) & (diff[0] <= -5):
        exemplars.append(value.iloc[[0]])
        
exemplars = pd.concat(exemplars)
exemplars['sgRNA_perturbation'] = exemplars['guide_identity'].str[:-16]
exemplars['sgRNA_target'] = exemplars['guide_identity'].str.split('_').str[0]
exemplars

In [ ]:
# get most transcriptomically similar cells with at least 2 UMIs
target_cell_num = 200
training_cells = {}
gene_idx = np.where(cancer.var['highly_variable'])[0]

for target in tqdm(exemplars['sgRNA_target'].unique()):
    local_exemp = exemplars[exemplars['sgRNA_target'] == target]
    local_valid = valid[valid['guide_identity'].str.startswith(target) & (valid['UMI_count'] > 1)]
    
    idx = np.where(cancer.obs.index.isin(local_exemp['cell_barcode']))[0]
    nnidx = np.where(cancer.obs.index.isin(local_valid['cell_barcode']))[0]
    
    XA = cancer[idx, gene_idx].obsm['X_pca']#.X
    XB = cancer[nnidx, gene_idx].obsm['X_pca']#.X
    
    dists = pairwise_distances(X=XA, Y=XB, metric='cosine')
    asidx = np.argsort(dists.mean(axis=0))

    train_idx = cancer.obs.index[nnidx[asidx[:target_cell_num]]]
    training_cells[target] = train_idx

In [ ]:
onehot = pd.DataFrame(index=cancer.obs.index)
for k in training_cells.keys():
    i = training_cells[k]
    
    onehot[k] = 0
    onehot[k].loc[i] = 1

onehot

In [ ]:
# pick only cells with one label
train_Y = onehot.loc[(onehot.sum(axis=1) == 1)]
train_Y

In [ ]:
labels = train_Y.columns[np.where(train_Y == 1)[1]]
labels

In [ ]:
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression

In [ ]:
# takes a while
# use PCA space instead of gene space

clf = LogisticRegressionCV()
clf.fit(cancer[train_Y.index, gene_idx].obsm['X_pca'], labels)

In [ ]:
from pickle import dump
with open('../results/PCA2KO_LRCV_model.pkl', 'wb') as f:
    dump(clf, f, protocol=4)

In [ ]:
proba = clf.predict_proba(cancer[:, gene_idx].obsm['X_pca'])
lr_prob = pd.DataFrame(proba, columns=clf.classes_, index=cancer.obs.index)
lr_prob

In [ ]:
lr_prob.to_csv('../results/PCA2KO_LRCV_prob.txt', sep='\t')

In [ ]:
# guide assignment
# 0. do not assign if no sgRNA detected
# 1. assign if only one sgRNA detected and has >= threshold UMI counts (umi_thr)
    # do not assign if < threshold UMI counts
# 2. assign to top sgRNA if difference is >= diff_thr for next highest UMI
# 3. last ditch assign if multiple sgRNAs detected: score based on logreg gex model, UMI counts, and guide_call prob
    # only for tie-breaking

umi_thr = 3  # must have this many UMIs minimum

outdir = '../results/guide_calls_sgRNA'
os.makedirs(outdir, exist_ok=True)

sg_assigns = {}
for diff_thr in np.arange(3, 11):
    print(diff_thr)
    assignments = {}
    method = {}
    
    outpath = os.path.join(outdir, f'sgRNA_assignments_{diff_thr}thr.txt')
    if os.path.exists(outpath):
        print('result exists, skipping')
        continue
    for group in tqdm(guide_calls.groupby('cell_barcode')):
        i, local = group
        if i not in lr_prob.index:
            continue
        local = local.sort_values(['UMI_count', 'proba'], ascending=False)
        delta = np.diff(local['UMI_count'])

        if len(local) == 0: # no sgRNA detected
            assignments[i] = 'unassigned'
            method[i] = 'no_sgRNA'

        elif len(local) == 1: # one sgRNA detected
            if local.iloc[0]['UMI_count'] >= umi_thr: # passes umi_thr
                assignments[i] = local.iloc[0]['guide_identity']
                method[i] = 'single_sgRNA_above_thr'
            else:  # does not pass umi_thr
                assignments[i] = 'unassigned'
                method[i] = 'single_sgRNA_below_thr'
        elif (local['UMI_count'] < umi_thr).all(): # do not assign if no sgRNA pass umi_thr
            assignments[i] = 'unassigned'
            method[i] = 'multiple_sgRNA_below_thr'
        elif delta[0] <= -diff_thr: # top guide is more abundant than next most abundant guide
            assignments[i] = local.iloc[0]['guide_identity']
            method[i] = 'multiple_sgRNA_above_diffthr'
        else: # last ditch assign based on logreg gex model
            # get most likely sgRNA per gene (remove UMI tag)
            try:
                max_sg_idx = local[local['UMI_count'] >= umi_thr].groupby('sgRNA_target')['proba'].nlargest(1).reset_index()['level_1'].values
            except KeyError:
                max_sg_idx = local[local['UMI_count'] >= umi_thr].groupby('sgRNA_target')['proba'].nlargest(1).reset_index()['index'].values
            # get most likely gene perturbation
            lr = lr_prob.loc[i]
            lr.index.name = 'sgRNA_target'

            # calculate score based on 
            # Poisson model proba * UMI counts * logreg model proba, taking top sgRNA
            scores = local.loc[max_sg_idx].set_index('sgRNA_target')['proba'] * local.loc[max_sg_idx].set_index('sgRNA_target')['UMI_count'] * lr
            gene_assign = scores.sort_values(ascending=False).index[0]
            assignments[i] = local[local['sgRNA_target'] == gene_assign].iloc[0]['guide_identity']
            method[i] = 'scoring'
    #         break
    sg_assign = pd.DataFrame.from_dict(assignments, orient='index', columns=['raw'])
    sg_assign['method'] = sg_assign.index.map(method)
    sg_assign['sgRNA_target'] = sg_assign['raw'].str.split('_').str[0]
    sg_assign['BC'] = sg_assign['raw'].str.split('_').str[-1]
    sg_assign['sgRNA_perturbation'] = sg_assign['sgRNA_target'] + '_' + sg_assign['BC'].str[:20]
    sg_assign['sgRNA_perturbation'] = sg_assign['sgRNA_perturbation'].replace('unassigned_unassigned', 'unassigned')
    sg_assign.drop(['raw', 'BC'], axis=1, inplace=True)

    sg_assign.to_csv(outpath, sep='\t')
    sg_assigns[diff_thr] = sg_assign
    

In [ ]:
compare = pd.DataFrame(index=sg_assigns[3].index)
for k in sg_assigns.keys():
    local = sg_assigns[k]
    compare[f'thr{k}_target'] = local['sgRNA_target']
    compare[f'thr{k}_sgRNA'] = local['sgRNA_perturbation']
compare

In [ ]:
# subset to cells that had differences in calling based on diff_thr
n_called = compare.loc[:, compare.columns.str.endswith('_sgRNA')].apply(lambda x: len(x.unique()), axis=1)
diff = compare.loc[n_called > 1].copy()
diff

In [ ]:
diff_props = {}

for c in diff.columns[diff.columns.str.endswith('_sgRNA')]:
    diff_props[c] = diff[c].value_counts() / len(diff)
diff_props = pd.concat(diff_props).reset_index().pivot(index='level_1', columns='level_0', values='count').fillna(0)
diff_props = diff_props.loc[:, natsorted(diff_props.columns)]
diff_props['non-diff'] = nondiff_props
diff_props = diff_props.fillna(0)
diff_props

In [ ]:
from scipy.stats import pearsonr, spearmanr

In [ ]:
pr2 = {}
sr2 = {}
for c in diff_props.columns:
    if c == 'non-diff': continue
    print(c)
    r, p = pearsonr(diff_props[c], diff_props['non-diff'])
    pr2[c] = r
    
    r, p = spearmanr(diff_props[c], diff_props['non-diff'])
    sr2[c] = r

In [ ]:
plt.plot(pr2.keys(), pr2.values(), label='pearsonr')
plt.plot(sr2.keys(), sr2.values(), label='spearmanr')
plt.legend()
plt.xticks(rotation=90)
plt.show()

In [ ]:
final_sg_assign = pd.read_table('../results/guide_calls_sgRNA/sgRNA_assignments_5thr.txt', index_col=0).reindex(cancer.obs.index)
final_sg_assign

In [ ]:
cancer.obs[['sgRNA_perturbation', 'sgRNA_target']] = final_sg_assign[['sgRNA_perturbation', 'sgRNA_target']]


In [ ]:
cancer.write_h5ad('../results/cancer_proc_singlets.h5ad')


# MAGeCK

In [ ]:
os.makedirs('../results/mageck')

In [ ]:
counts = cancer.obs[['sgRNA_perturbation', 'tumor']].dropna().value_counts()
counts_reps = counts.reset_index().pivot(index='sgRNA_perturbation', columns='tumor', values=0).replace(np.nan, 0).astype(int)
counts_reps.drop('unassigned', inplace=True)

counts_reps

In [ ]:
invitro = pd.read_table('../data/invitro_barcode_counts.txt.gz', index_col=0)
invitro

In [ ]:
mageck_counts = pd.concat([counts_reps, invitro], axis=1).fillna(0).astype(int)

mageck_counts.insert(0, 'sgRNA_target', mageck_counts.index.str.split('_').str[0])
mageck_counts = mageck_counts.reset_index().set_index(['sgRNA_perturbation', 'sgRNA_target'])
mageck_counts.columns.name = None

mageck_counts

In [ ]:
mageck_counts.to_csv('../results/mageck/mageck_perturb_cell_counts.txt', sep='\t')


In [ ]:
# echo safe > ../results/mageck/safe.txt
# cd ../results/mageck

# in vivo vs input
# mageck test -k mageck_perturb_cell_counts.txt -c invitro_input -t 1N,1L,1LR,1RR,2N,2L,2R,2LR,2RR -n invivo_v_input --norm-method control --control-gene safe.txt --additional-rra-parameters "--permutation 10000" --pdf-report --normcounts-to-file

# output vs input
# mageck test -k mageck_perturb_cell_counts.txt -c invitro_input -t invitro_output -n output_v_input --control-gene safe.txt --additional-rra-parameters "--permutation 10000" --pdf-report --normcounts-to-file

In [ ]:
mageck_rra = pd.read_table('../results/mageck/invivo_v_input.gene_summary.txt', index_col=0)
mageck_rra['neg|logfdr'] = -np.log10(mageck_rra['neg|fdr'])
mageck_rra['pos|logfdr'] = -np.log10(mageck_rra['pos|fdr'])
mageck_rra['neg|logscore'] = -np.log10(mageck_rra['neg|score'])
mageck_rra['pos|logscore'] = -np.log10(mageck_rra['pos|score'])

mageck_rra

In [ ]:
assert (mageck_rra['neg|lfc'] == mageck_rra['pos|lfc']).all()
x = mageck_rra['neg|lfc']
y = mageck_rra[['neg|logscore', 'pos|logscore']].max(axis=1)

def size_func(x):
    x = np.array(x)
    return (-np.log10(x)).clip(min=0.2)*50

s = size_func(mageck_rra[['neg|fdr', 'pos|fdr']].min(axis=1))
c = np.array(['red' if a < 0.05 else 'blue' if b < 0.05 else 'grey' for a, b in zip(mageck_rra['neg|fdr'], mageck_rra['pos|fdr'])])

with plt.rc_context({'figure.dpi':300, 'figure.figsize': (5, 4)}):
    plt.scatter(x, y, 
                c=c, 
                s=s,
                edgecolor='k', lw=0.25)
    
    idx = np.where(c != 'grey')[0]
    ylb = y.iloc[idx]
    xlb = x.iloc[idx]
    texts =[plt.text(x, y, i) for i, x, y in zip(ylb.index, xlb, ylb)]
    adjust_text(texts)

    plt.xlabel('mean fold change (log2)')
    plt.ylabel('MAGeCK score (-log10)')
    plt.title('Cancer cell in vivo vs input')

    sizes = (0.2, 0.05, 0.001)
    markers = [plt.scatter([], [], s=size_func(s), marker='o', c='grey', edgecolor='k', lw=0.25) for s in sizes]
    plt.legend(markers, sizes, scatterpoints=1, ncol=1, fontsize=8, title='FDR', loc='lower left', labelspacing=.75)
    plt.show()

In [ ]:
mageck_invitro = pd.read_table('../results/mageck/output_v_input.gene_summary.txt', index_col=0)

mageck_invitro['neg|logfdr'] = -np.log10(mageck_invitro['neg|fdr'])
mageck_invitro['pos|logfdr'] = -np.log10(mageck_invitro['pos|fdr'])
mageck_invitro['neg|logscore'] = -np.log10(mageck_invitro['neg|score'])
mageck_invitro['pos|logscore'] = -np.log10(mageck_invitro['pos|score'])

mageck_invitro

In [ ]:
y = mageck_rra['neg|lfc']
x = mageck_invitro['neg|lfc'].loc[y.index]

def size_func(x):
    x = np.array(x)
    return (-np.log10(x)).clip(min=0.2)*70

c = np.array(['red' if a < 0.05 else 'blue' if b < 0.05 else 'grey' for a, b in zip(mageck_rra['neg|fdr'], mageck_rra['pos|fdr'])])
s = size_func(mageck_rra[['neg|fdr', 'pos|fdr']].min(axis=1))

with plt.rc_context({'figure.dpi':200, 'figure.figsize': (5, 4)}):
    plt.scatter(x, y, 
                s=s,
                c=c, 
                edgecolor='k', lw=0.25)
    
    idx = np.where(c != 'grey')[0]
    ylb = y.iloc[idx]
    xlb = x.iloc[idx]
    texts =[plt.text(x, y, i) for i, x, y in zip(ylb.index, xlb, ylb)]
    adjust_text(texts)

    plt.xlabel('in vitro output vs input LFC')
    plt.ylabel('in vivo vs input LFC')
    plt.axvline(0, color='k', lw=1)
    plt.axhline(0, color='k', lw=1)
    plt.title('in vivo vs in vitro enrichments')

    sizes = (0.2, 0.05, 0.001)
    markers = [plt.scatter([], [], s=size_func(s), marker='o', c='grey', edgecolor='k', lw=0.25) for s in sizes]
    plt.legend(markers, sizes, scatterpoints=1, ncol=1, fontsize=8, title='In vivo FDR', loc='lower right', labelspacing=.75)

    plt.show()

# DEG vs safe

In [ ]:
# run script
!python ../auxiliary_scripts/logreg_de.py --h5ad ../results/cancer_proc_singlets.h5ad --label sgRNA_target --layer sf_log1p --compare Adar,safe Angpt1,safe Angpt2,safe Angptl2,safe Angptl4,safe Angptl6,safe Bmp1,safe Bmp2,safe Bmp4,safe Ccl2,safe Ccl25,safe Ccl5,safe Ccl7,safe Ccl8,safe Ccm2,safe Ccr10,safe Ccr5,safe Ccrl2,safe Cd274,safe Cd276,safe Cd40,safe Cd47,safe Cd48,safe Cd80,safe Ceacam1,safe Ciita,safe Cklf,safe Clcf1,safe Clec2d,safe Clec4g,safe Cmtm3,safe Cmtm7,safe Cntf,safe Crlf1,safe Csf1,safe Csf2,safe Csf3,safe Ctf1,safe Ctf2,safe Cx3cl1,safe Cxcl1,safe Cxcl10,safe Cxcl2,safe Cxcl5,safe Cxcl9,safe Enpp1,safe Flt3l,safe Gdf11,safe Gdf15,safe Grem1,safe Hmgb1,safe Icam1,safe Icam2,safe Ifnar1,safe Ifnar2,safe Ifngr1,safe Ifngr2,safe Il11,safe Il18,safe Il24,safe Il33,safe Il34,safe Il4,safe Il6,safe Kitl,safe Krit1,safe Lefty1,safe Lgals3,safe Lgals9,safe Lif,safe Ly9,safe Mif,safe Ncam1,safe Nectin1,safe Nectin2,safe Nectin3,safe Pdcd10,safe Ppbp,safe Pvr,safe Raet1d,safe Raet1e,safe Rela,safe Slamf8,safe Smad1,safe Smad2,safe Smad3,safe Smad4,safe Smad5,safe Smad7,safe Spp1,safe Stat1,safe Stat2,safe Stat3,safe Stat5a,safe Stat5b,safe Stat6,safe Tgfb1,safe Tgfb2,safe Tgfb3,safe Tgfbi,safe Tgfbr1,safe Tgfbr2,safe Tgfbr3,safe Tnfrsf10b,safe Tnfrsf11a,safe Tnfrsf12a,safe Tnfrsf14,safe Tnfrsf1a,safe Tnfrsf1b,safe Tnfrsf21,safe Tnfsf10,safe Tnfsf12,safe Tnfsf13,safe Tnfsf9,safe Tradd,safe Traf1,safe Traf2,safe Traf3,safe Traf3ip1,safe Traf5,safe Traf6,safe Traf7,safe Trafd1,safe Ulbp1,safe Vcam1,safe Vegfa,safe Vegfb,safe Vegfc,safe Vegfd,safe Wnt10b,safe Wnt5a,safe Wnt7b,safe

In [ ]:
# move to results folder

# cNMF
run with NVIDIA L4 GPU on Google Cloud instance

In [ ]:
import os, sys

import torch
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns

import pickle
from glob import glob
from natsort import natsorted
from copy import deepcopy
from tqdm import tqdm

In [ ]:
sys.path.append('../auxiliary_scripts/NMF_gpu')
from NMF.nmf import StandardNMF
from NMF.sweep import run_rank_sweep
from NMF.graphics import plot_pairwise_component_distances, plot_all_clustered_components, plot_sweep_metrics

In [ ]:
def reconst_pos_neg(loadings, scale=True):
    loadings_reconst = np.zeros((len(loadings)//2, loadings.shape[1]))

    pos_tmp = loadings.loc[loadings.index.str.startswith('pos::')]
    neg_tmp = loadings.loc[loadings.index.str.startswith('neg::')]
        
    pos_idx = np.where(pos_tmp.values > neg_tmp.values)
    neg_idx = np.where(neg_tmp.values > pos_tmp.values)
    
    if scale:
        pos_tmp = pos_tmp / pos_tmp.max()
        neg_tmp = neg_tmp / neg_tmp.max()

    loadings_reconst[pos_idx] = pos_tmp.values[pos_idx]
    loadings_reconst[neg_idx] = -neg_tmp.values[neg_idx]

    loadings_reconst = pd.DataFrame(loadings_reconst, index=pos_tmp.index.str.split("::").str[-1])
    loadings_reconst.index.name = None
    
    return loadings_reconst

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
infiles = natsorted(glob('../results/logreg_de_cancer/*lrLRT.txt'))
len(infiles)

In [ ]:
stack = []

for f in tqdm(infiles):
    target = os.path.basename(f).split('_safe')[0]
    local = pd.read_table(f, index_col=0)
    local.index.name = 'gene'
    local['target'] = target
    stack.append(local.reset_index().copy())
    
deg_results = pd.concat(stack)
deg_results['signed_logfdr'] = np.sign(deg_results['mean']) * -np.log10(deg_results['fdr'])
deg_results

In [ ]:
# rows are features
# columns are samples

slogf = deg_results.pivot(index='gene', columns='target', values='signed_logfdr')

syn_genes = slogf.index[slogf.index.str.startswith(('syn_', 'syn-'))]
slogf = slogf.drop(syn_genes)

# z-score
slogf = (slogf) / slogf.std().replace(0, 1)

slogf

In [ ]:
nmf_outdir = '../results/cNMF_cancer'
os.makedirs(outdir, exist_ok=True)

In [ ]:
targets = slogf.columns

len(targets)

In [ ]:
pos_idx = np.where(slogf.values > 0)
neg_idx = np.where(slogf.values < 0)

pos_slogf = np.zeros_like(slogf)
neg_slogf = np.zeros_like(slogf)

pos_slogf[pos_idx] = slogf.values[pos_idx]
neg_slogf[neg_idx] = np.abs(slogf.values[neg_idx])

split_slogf = pd.DataFrame(np.vstack([pos_slogf, neg_slogf]),
                           columns=slogf.columns, 
                           index=list('pos::' + slogf.index) + list('neg::' + slogf.index))

split_slogf

In [ ]:
pd.Series(split_slogf.columns).to_csv(os.path.join(nmf_outdir, 'targets.txt'), header=None, index=False, sep='\t')
pd.Series(split_slogf.index).to_csv(os.path.join(nmf_outdir, 'genes.txt'), header=None, index=False, sep='\t')
np.save(os.path.join(nmf_outdir, 'input_matrix.npy'), split_slogf.values)

In [ ]:
rank_range = np.arange(2, 31)

outpath = os.path.join(nmf_outdir, 'cancer_cNMF_rank_sweep_results.pkl')

rank_results = run_rank_sweep(split_slogf.values, 
                            model_cls=StandardNMF,
                            rank_range=rank_range, 
                            n_runs=100, 
                            device=device)

with open(outpath, 'wb') as h:
    pickle.dump(rank_results, h)

In [ ]:
# Plot Metrics
plot_sweep_metrics(rank_results)

In [ ]:
import numpy as np
import scipy.stats as stats

def lstsq_pvalues(X, Y):
    """
    X: ndarray of shape (n, k)
    Y: ndarray of shape (n, i)
    """
    X = np.asarray(X)
    Y = np.asarray(Y)
    
    n, k = X.shape
    _, i = Y.shape
    dof = n - k
    
    beta, resid, rank, sing = np.linalg.lstsq(X, Y, rcond=None)
    
    Y_pred = X @ beta
    ss_resid = np.sum((Y - Y_pred) ** 2, axis=0)
    sigma_sq = ss_resid / dof  # shape (i,)
    
    XtX_inv = np.linalg.pinv(X.T @ X)
    diag_XtX_inv = np.diag(XtX_inv)  # shape (k,)
    
    se = np.sqrt(np.outer(diag_XtX_inv, sigma_sq))
    t_stat = beta / se
    p_values = 2 * stats.t.sf(np.abs(t_stat), df=dof)
    
    return beta, resid, p_values

In [ ]:
k = 15
print(f'k={k}')


genes = pd.read_table(os.path.join(nmf_outdir, 'genes.txt'), header=None).squeeze().values
targets = index = pd.read_table(os.path.join(nmf_outdir, 'targets.txt'), header=None).squeeze().values

input_matrix = np.load(os.path.join(nmf_outdir, f'input_matrix.npy'))
tmp = pd.DataFrame(input_matrix, index=genes, columns=targets)
input_reconst = reconst_pos_neg(tmp, scale=False)
input_reconst.columns = targets

gene_loadings = pd.DataFrame(rank_results[k]['W'], index=genes)
gene_loadings.index.name = None
factor_loadings = pd.DataFrame(rank_results[k]['H'].T, index=index)
factor_loadings.index.name = None

gene_loadings_reconst = reconst_pos_neg(gene_loadings, scale=False)

# gene OLS
X = factor_loadings.values
Y = input_reconst.values.T
beta, resid, pval = lstsq_pvalues(X, Y)
gene_beta = pd.DataFrame(beta, columns=input_reconst.index).T
gene_resid = pd.Series(resid, index=input_reconst.index)
gene_pval = pd.DataFrame(pval, columns=input_reconst.index).T

# factor OLS
X = gene_loadings_reconst.values
Y = input_reconst.values
beta, resid, pval = lstsq_pvalues(X, Y)
factor_beta = pd.DataFrame(beta, columns=targets).T
factor_resid = pd.Series(resid, index=targets)
factor_pval = pd.DataFrame(pval, columns=targets).T


gene_loadings.to_csv(os.path.join(nmf_outdir, f'k{k}_gene_loadings.txt'), sep='\t')
factor_loadings.to_csv(os.path.join(nmf_outdir, f'k{k}_factor_loadings.txt'), sep='\t')
gene_loadings_reconst.to_csv(os.path.join(nmf_outdir, f'k{k}_gene_loadings_reconst.txt'), sep='\t')

factor_beta.to_csv(os.path.join(nmf_outdir, f'k{k}_OLS_factor_beta.txt'), sep='\t')
factor_resid.to_csv(os.path.join(nmf_outdir, f'k{k}_OLS_factor_resid.txt'), sep='\t')
factor_pval.to_csv(os.path.join(nmf_outdir, f'k{k}_OLS_factor_pval.txt'), sep='\t')

gene_beta.to_csv(os.path.join(nmf_outdir, f'k{k}_OLS_gene_beta.txt'), sep='\t')
gene_resid.to_csv(os.path.join(nmf_outdir, f'k{k}_OLS_gene_resid.txt'), sep='\t')
gene_pval.to_csv(os.path.join(nmf_outdir, f'_k{k}_OLS_gene_pval.txt'), sep='\t')

## pathway enrichment

In [ ]:
import gseapy as gp

In [ ]:
background_set = cancer.var.index[cancer.X.getnnz(axis=0) > 0]
background_set = background_set[~background_set.str.startswith('syn_')].tolist()
len(background_set)

In [ ]:
go_bp = gp.get_library('GO_Biological_Process_2025', organism='mouse')

In [ ]:
from kneed import KneeLocator

def find_knee(rnk, verbose=False):
    rnk = rnk.sort_values()
    
    zidx = np.where(rnk == 0)[0]
    if len(zidx) == 0:
        idx = np.argmin(rnk.abs())
        zidx = [idx, idx]

    # negative
    y = rnk.values[:zidx[0]]  # first zero value
    x = np.arange(len(y))
    kneedle = KneeLocator(x, y, S=1.0, curve='concave', direction='increasing')
    n1 = kneedle.knee

    # positive
    y = rnk.values[zidx[-1]:] # last zero value
    x = np.arange(zidx[-1], len(rnk))
    kneedle = KneeLocator(x, y, S=1.0, curve='convex', direction='increasing')
    n2 = kneedle.knee

    dn_genes = rnk.index[:n1].tolist()
    up_genes = rnk.index[n2:].tolist()
    if verbose: print(len(dn_genes), len(up_genes))
    return dn_genes, up_genes
    

In [ ]:
deg_results['rnk_stat'] = np.sign(deg_results['mean']) * -np.log10('log_loss')

In [ ]:
cancer_knee_genes = {}

for group in tqdm(deg_results.groupby('target')):
    t, local = group
    dn, up = find_knee(local['rnk_stat'].sort_values())
    up = up[::-1]
    
    cancer_knee_genes[f'{t}_up'] = up
    cancer_knee_genes[f'{t}_down'] = dn
len(cancer_knee_genes)

In [ ]:
# write
write_gmt(cancer_knee_genes, '../results/cancer_kneedle.gmt')

In [ ]:
all_res = []
for key in cancer_knee_genes.keys():
    genes = cancer_knee_genes[key]
    factor, direction = factor.split('_')
    res = gp.enrich(genes, 
                go_bp, 
                outdir=None, 
                background=background_set,
                verbose=False)


    res_df = res.res2d.sort_values('P-value')
    res_df['direction'] = direction
    res_df['NMF_factor'] = factor
    
    or_sign = res_df['direction'].replace({'down': -1, 'up': 1})
    res_df['signed_OR'] = or_sign * res_df['Odds Ratio']
    res_df['signed_log2OR'] = or_sign * np.log2(res_df['Odds Ratio'])
    res_df['slogpadj'] = or_sign * -np.log10(res_df['Adjusted P-value'])
    
    all_res.append(res_df.copy())
gobp_res = pd.concat(all_res)

In [ ]:
# write
gobp_res.to_csv(os.path.join(nmf_outdir, 'cancer_cNMF_GOBP_fisher.txt'), sep='\t')

In [ ]:
filtered_res = gobp_res.copy()

filtered_res['n_genes'] = filtered_res['Overlap'].str.split('/').str[0].astype(int)
filtered_res['n_set'] = filtered_res['Overlap'].str.split('/').str[1].astype(int)
filtered_res = filtered_res[(filtered_res['n_genes']>1) 
                            & (filtered_res['n_set'] < 200)]

sig_mat = filtered_res.groupby(['Term', 'NMF_factor'])['slogpadj'] \
                   .agg(get_max_abs) \
                   .unstack(fill_value=0)
or_mat = filtered_res.groupby(['Term', 'NMF_factor'])['signed_log2OR'] \
                   .agg(get_max_abs) \
                   .unstack(fill_value=0)


or_mat = or_mat.loc[sig_mat.index, sig_mat.columns]

In [ ]:
def get_jaccard(str_a, str_b):
    """Calculates Jaccard Index between two semicolon-delimited strings."""
    if not isinstance(str_a, str) or not isinstance(str_b, str):
        return 0.0
    set_a = set(str_a.split(';'))
    set_b = set(str_b.split(';'))
    union_len = len(set_a | set_b)
    return len(set_a & set_b) / union_len if union_len > 0 else 0.0

def get_refined_exclusive_pathways(df, 
                                   rnk_stat_colnames=['slogpadj'], 
                                   factor_colname='NMF_factor',
                                   direction_colname='direction',
                                   term_colname='Term',
                                   gene_colname='Genes',
                                   n=3, jaccard_threshold=0.5):
    """
    Assigns pathways that are both mutually exclusive across factors 
    and non-redundant within a factor's own signature.
    
    Expected columns in df: 
    ['factor', 'pathway_name', 'direction', 'enrichment_statistic', 'genes']
    """
    # sort by absolute magnitude
    pool = df.copy()
    abs_colnames = [_+'_abs' for _ in rnk_stat_colnames]
    pool[abs_colnames] = pool[rnk_stat_colnames].abs()
    pool = pool.sort_values(abs_colnames, ascending=False)

    assigned_globally = set()
    # Results store: { (factor, direction): [ (pathway_name, gene_str), ... ] }
    results = { (f, d): [] for f in df[factor_colname].unique() for d in ['up', 'down'] }
    
    for _, row in pool.iterrows():
        f, d = row[factor_colname], row[direction_colname]
        p_name, p_genes = row[term_colname], row[gene_colname]
        
        # mutual exclusivity
        if len(results[(f, d)]) < n and p_name not in assigned_globally:
            
            # jaccard
            is_redundant = False
            for _, existing_genes in results[(f, d)]:
                if get_jaccard(p_genes, existing_genes) > jaccard_threshold:
                    is_redundant = True
                    break
            
            if not is_redundant:
                results[(f, d)].append((p_name, p_genes))
                assigned_globally.add(p_name)
            
    # formatting
    final_rows = []
    for (f, d), pathways in results.items():
        for p_name, _ in pathways:
            orig_row = df[(df[factor_colname] == f) & (df[term_colname] == p_name)].iloc[0]
            final_rows.append(orig_row)
            
    return pd.DataFrame(final_rows), results

In [ ]:
topn = 2
top_pathways, top_dict = get_refined_exclusive_pathways(filtered_res, n=topn, jaccard_threshold=0.5,
                                              rnk_stat_colnames=['slogpadj', 'signed_log2OR'],
                                              factor_colname='NMF_factor',
                                              direction_colname='direction',
                                              term_colname='Term',
                                              gene_colname='Genes',
                                              )
len(top_pathways)

In [ ]:
dotplot(color_df=c_df, size_df=s_df, 
        ygroup=gs_groups,
        size_scale=2,
        figsize_xscale=0.65,
        figsize_yscale=0.2,
        size_legend_title='Log2 odds ratio',
        size_legend_vmin=1, size_legend_vmax=10,
        edgecolor='k', edgewidth=0.5,
        cmap='vlag', center=0, show=True)
